# 03 · Join Sofascore + Capology — France Ligue 1 20/21

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2020/21 de Ligue 1 francesa**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_france_2021.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_france_2021.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  573 jugadores | 116 columnas
Capology:   586 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   as monaco
   nimes olympique
   olympique de marseille
   olympique lyonnais
   paris saint germain
   rc lens
   rc strasbourg
   saint etienne
   stade brestois
   stade de reims
   stade rennais

En Capology pero no en Sofascore:
   brest
   lens
   lyon
   marseille
   monaco
   nimes
   psg
   reims
   rennes
   st etienne
   strasbourg


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [6]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'brest':'stade brestois',
            'lens':'rc lens',
            'lyon':'olympique lyonnais',
            'marseille':'olympique de marseille',
            'monaco':'as monaco',
            'nimes':'nimes olympique',
            'psg':'paris saint germain',
            'reims':'stade de reims',
            'rennes':'stade rennais',
            'st etienne':'saint etienne',
            'strasbourg':'rc strasbourg'
}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')


✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [7]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 479/573 (83.6%)
Sin emparejar: 94


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [8]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          10
Revisión media    (0.75 ≤ score < 0.90):   5
Revisión estricta (0.50 ≤ score < 0.75):   43
Revisión muy est. (score < 0.50):           36


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [9]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
50,Billal Benkhedim,Saint-Étienne,bilal benkhedim,0.968
44,Radosław Majecki,AS Monaco,radoslaw majecki,0.968
54,Hianga'a M'Bock,Stade Brestois,hianga a mbock,0.966
67,Samy Benchama,Montpellier,samy benchamma,0.963
23,Stanley N'Soki,Nice,stanley nsoki,0.963
14,Steven​ N'Zonzi,Stade Rennais,steven nzonzi,0.963
21,Didier N'Dong,Dijon,didier ndong,0.960
93,Yaya Soumare,Olympique Lyonnais,yahya soumare,0.960
1,Burak Yılmaz,Lille,burak yilmaz,0.957
9,Yusuf Yazıcı,Lille,yusuf yazici,0.909


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [10]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
69,Ahmad Toure Ngouyamsa,Dijon,ahmad ngouyamsa,0.833
17,Djamel Benlamri,Olympique Lyonnais,djamel eddine benlamri,0.811
41,Florentino Luís,AS Monaco,florentino,0.800
0,Marçal,Olympique Lyonnais,marcelo,0.769
8,Pape Matar Sarr,Metz,pape sarr,0.750


In [11]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = ['marcal'

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')


Aceptados: 4 | Excluidos: 1


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [12]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
64,Hicham Mahou,Nice,hicham boudaoui,0.741
34,Vagner Dias,Metz,vagner,0.706
25,David Pereira da Costa,RC Lens,david costa,0.667
30,Abdoulaye Sylla,Nantes,abdoulaye toure,0.667
4,Boubakar Kouyaté,Metz,boubacar traore,0.645
76,Yann M'Vila,Saint-Étienne,yvann macon,0.636
33,Edson Mexer,Bordeaux,mexer,0.625
40,Junior Dina Ebimbe,Dijon,eric ebimbe,0.621
85,Matthis Abline,Stade Rennais,romain salin,0.615
35,Ahmadou Bamba Dieng,Olympique de Marseille,aaron kamardin,0.606


In [13]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['vagner dias',
                    'david pereira da costa',
                    'edson mexer',
                    'junior dina ebimbe',
                    'danilo barbosa',
                    'rafinha alcantara',
                    'vitorino hilton'

]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')


Aceptados del nivel bajo: 7


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [14]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
73,Cheick Tidiane Sabaly,Metz,victorien angban,0.486
31,Joachim Andersen,Olympique Lyonnais,thiago mendes,0.483
59,Eddy Sylvestre,Nice,teddy boulhendi,0.483
60,Aurélien Scheidler,Dijon,pape cheikh,0.483
52,Anthony Modeste,Saint-Étienne,aimen moueffek,0.483
16,Sékou Mara,Bordeaux,youssouf sabaly,0.480
80,Jesé Rodríguez,Paris Saint-Germain,sergio rico,0.480
65,Rubén Pardo,Bordeaux,hatem ben arfa,0.480
72,Andy Diouf,Stade Rennais,jeremy doku,0.476
24,Issouf Sissokho,Bordeaux,youssouf sabaly,0.467


In [15]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')


Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [16]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 500/573 (87.3%)
Sin salario:     73


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [17]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 73


,player,team,minutesPlayed,appearances,goals,assists
0,Giulian Biancone,AS Monaco,92,2,0,0
1,Baptiste Santamaria,Angers,270,3,0,0
2,Rayan Aït-Nouri,Angers,187,3,0,1
3,Wilfried Kanga,Angers,134,2,0,0
4,Casimir Ninga,Angers,22,1,0,0
5,Yassin Fortune,Angers,14,1,0,0
6,Rachid Alioui,Angers,14,1,0,0
7,Josh Maja,Bordeaux,815,17,2,0
8,Jimmy Briand,Bordeaux,557,24,1,2
9,Sékou Mara,Bordeaux,232,8,1,2


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [18]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  AS Monaco  —  SF sin salario:


,player,minutesPlayed
0,Giulian Biancone,92


  CG plantilla completa:


,player,player_norm
0,Aleksandr Golovin,aleksandr golovin
1,Aurélien Tchouaméni,aurelien tchouameni
2,Axel Disasi,axel disasi
3,Benjamin Lecomte,benjamin lecomte
4,Benoît Badiashile,benoit badiashile
5,Caio Henrique,caio henrique
6,Cesc Fàbregas,cesc fabregas
7,Chrislain Matsima,chrislain matsima
8,Djibril Sidibé,djibril sidibe
9,Eliot Matazo,eliot matazo



  Angers  —  SF sin salario:


...

,player,player_norm
0,Adrien Hunou,adrien hunou
1,Adrien Truffert,adrien truffert
2,Alfred Gomis,alfred gomis
3,Benjamin Bourigeaud,benjamin bourigeaud
4,Brandon Soppy,brandon soppy
5,Clément Grenier,clement grenier
6,Dalbert,dalbert
7,Damien Da Silva,damien da silva
8,Daniele Rugani,daniele rugani
9,Eduardo Camavinga,eduardo camavinga



  Stade de Reims  —  SF sin salario:


,player,minutesPlayed
0,Alexis Flips,41
1,Dion Lopy,256
2,Hugo Ekitiké,86
3,Moise Sakava,8
4,Tristan Dingomé,14


  CG plantilla completa:


,player,player_norm
0,Anastasios Donis,anastasios donis
1,Arbër Zeneli,arber zeneli
2,Boulaye Dia,boulaye dia
3,Dario Maresic,dario maresic
4,Dereck Kutesa,dereck kutesa
5,Dialy Ndiaye,dialy ndiaye
6,El Bilal Touré,el bilal toure
7,Fodé Doucouré,fode doucoure
8,Fraser Hornby,fraser hornby
9,Ghislain Konan,ghislain konan


In [19]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {
    ('boubakar kouyate', 'metz'): ('kiki kouyate', 'metz'),
}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')


Matches manuales definidos: 1


In [20]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')

✅ Match manual aplicado: boubakar kouyate (metz) → kiki kouyate (metz)

Tras matches manuales: 501/573 (87.4%)


In [21]:
pd.reset_option('display.max_rows')

## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [22]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_france_2021.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_france_2021.csv
   Jugadores totales:  573
   Con salario:        501
   Sin salario (NaN):  72
   Columnas:           121
